## Zadania

**Zadanie 1**

In [3]:
import requests
import time
import concurrent.futures

In [4]:
CAT_API_URL = "https://catfact.ninja/fact"

def pobierz_fakt():
    odpowiedz = requests.get(CAT_API_URL, timeout=10)
    odpowiedz.raise_for_status()
    return odpowiedz.json().get("fact")

In [5]:
#1. pobieranie sekwencyjne
start = time.perf_counter()

fakty_po_kolei = []

for i in range(20):
    fakt = pobierz_fakt()
    fakty_po_kolei.append(fakt)

koniec = time.perf_counter()
czas_sekwencyjnie = koniec - start

print("FAKTY POBRANE SEKWENCYJNIE:")
for nr, fakt in enumerate(fakty_po_kolei, start=1):
    print(f"{nr}. {fakt}")

print(f"\nCzas sekwencyjnie: {czas_sekwencyjnie:.2f} s")

FAKTY POBRANE SEKWENCYJNIE:
1. Unlike humans, cats are usually lefties. Studies indicate that their left paw is typically their dominant paw.
2. Abraham Lincoln loved cats. He had four of them while he lived in the White House.
3. Miacis, the primitive ancestor of cats, was a small, tree-living creature of the late Eocene period, some 45 to 50 million years ago.
4. The leopard is the most widespread of all big cats.
5. Cats make about 100 different sounds. Dogs make only about 10.
6. You check your cats pulse on the inside of the back thigh, where the leg joins to the body. Normal for cats: 110-170 beats per minute.
7. Both humans and cats have identical regions in the brain responsible for emotion.
8. A cat can sprint at about thirty-one miles per hour.
9. The cat's clavicle, or collarbone, does not connect with other bones but is buried in the muscles of the shoulder region. This lack of a functioning collarbone allows them to fit through any opening the size of their head.
10. The l

In [6]:
#2. pobieranie wielowątkowe
start = time.perf_counter()

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    fakty_watki = list(executor.map(lambda _: pobierz_fakt(), range(20)))

koniec = time.perf_counter()
czas_wielowatkowo = koniec - start

print("FAKTY POBRANE WIELOWĄTKOWO:")
for nr, fakt in enumerate(fakty_watki, start=1):
    print(f"{nr}. {fakt}")

print(f"\nCzas wielowątkowo: {czas_wielowatkowo:.2f} s")

FAKTY POBRANE WIELOWĄTKOWO:
1. While it is commonly thought that the ancient Egyptians were the first to domesticate cats, the oldest known pet cat was recently found in a 9,500-year-old grave on the Mediterranean island of Cyprus. This grave predates early Egyptian art depicting cats by 4,000 years or more.
2. A cat's normal temperature varies around 101 degrees Fahrenheit.
3. If a cat is frightened, the hair stands up fairly evenly all over the body; when the cat is threatened or is ready to attack, the hair stands up only in a narrow band along the spine and tail.
4. Polydactyl cats (a cat with 1-2 extra toes on their paws) have this as a result of a genetic mutation. These cats are also referred to as 'Hemingway cats' because writer Ernest Hemingway reportedly owned dozens of them at his home in Key West, Florida.
5. On average, a cat will sleep for 16 hours a day.
6. Your cat's heart beats at a rate almost double that of yours, from 110-140 beats per minute.
7. A cat’s nose pad is

In [7]:
#3. porównanie
print("PORÓWNANIE:")
print(f"Sekwencyjnie:   {czas_sekwencyjnie:.2f} s")
print(f"Wielowątkowo:   {czas_wielowatkowo:.2f} s")

if czas_wielowatkowo < czas_sekwencyjnie:
    roznica = czas_sekwencyjnie - czas_wielowatkowo
    print(f"Wielowątkowo było szybciej o około {roznica:.2f} s.")
else:
    roznica = czas_wielowatkowo - czas_sekwencyjnie
    print(f"Sekwencyjnie było szybciej o około {roznica:.2f} s.")

PORÓWNANIE:
Sekwencyjnie:   6.35 s
Wielowątkowo:   0.35 s
Wielowątkowo było szybciej o około 6.00 s.


**Zadanie 2**

In [8]:
import queue
import threading

In [9]:
kolejka = queue.Queue()

ILE_LICZB = 20
STOP = None

In [10]:
def producent():
    for liczba in range(1, ILE_LICZB + 1):
        kolejka.put(liczba)
        print(f"Producent wrzucił: {liczba}")
        time.sleep(0.05)

    #2x stop bo dwóch konsumentów
    kolejka.put(STOP)
    kolejka.put(STOP)

In [11]:
def konsument_parzyste():
    while True:
        liczba = kolejka.get()

        if liczba is STOP:
            kolejka.task_done()
            print("Konsument parzystych kończy pracę")
            break

        if liczba % 2 == 0:
            print(f"  Konsument PARZYSTE pobrał: {liczba}")
            kolejka.task_done()
            time.sleep(0.1)
        else:
            kolejka.task_done()
            kolejka.put(liczba)
            time.sleep(0.02)

In [12]:
def konsument_nieparzyste():
    while True:
        liczba = kolejka.get()

        if liczba is STOP:
            kolejka.task_done()
            print("Konsument nieparzystych kończy pracę")
            break

        if liczba % 2 != 0:
            print(f"  Konsument NIEPARZYSTE pobrał: {liczba}")
            kolejka.task_done()
            time.sleep(0.1)
        else:
            kolejka.task_done()
            kolejka.put(liczba)
            time.sleep(0.02)

In [13]:
watek_producent = threading.Thread(target=producent)
watek_parzyste = threading.Thread(target=konsument_parzyste)
watek_nieparzyste = threading.Thread(target=konsument_nieparzyste)

watek_producent.start()
watek_parzyste.start()
watek_nieparzyste.start()

watek_producent.join()
watek_parzyste.join()
watek_nieparzyste.join()

print("Koniec programu")

Producent wrzucił: 1
  Konsument NIEPARZYSTE pobrał: 1
Producent wrzucił: 2
  Konsument PARZYSTE pobrał: 2
Producent wrzucił: 3
  Konsument NIEPARZYSTE pobrał: 3
Producent wrzucił: 4
  Konsument PARZYSTE pobrał: 4
Producent wrzucił: 5
  Konsument NIEPARZYSTE pobrał: 5
Producent wrzucił: 6  Konsument PARZYSTE pobrał: 6

Producent wrzucił: 7
  Konsument NIEPARZYSTE pobrał: 7
Producent wrzucił: 8
  Konsument PARZYSTE pobrał: 8
Producent wrzucił: 9  Konsument NIEPARZYSTE pobrał: 9

Producent wrzucił: 10
  Konsument PARZYSTE pobrał: 10
Producent wrzucił: 11
  Konsument NIEPARZYSTE pobrał: 11
Producent wrzucił: 12
  Konsument PARZYSTE pobrał: 12
Producent wrzucił: 13
  Konsument NIEPARZYSTE pobrał: 13
Producent wrzucił: 14
  Konsument PARZYSTE pobrał: 14
Producent wrzucił: 15
  Konsument NIEPARZYSTE pobrał: 15
Producent wrzucił: 16
  Konsument PARZYSTE pobrał: 16
Producent wrzucił: 17
  Konsument NIEPARZYSTE pobrał: 17
Producent wrzucił: 18
  Konsument PARZYSTE pobrał: 18
Producent wrzucił: 

**Zadanie 3**

In [18]:
import multiprocessing
from lab2_functions import calculate_power_sum

In [20]:
if __name__ == "__main__":
    zakres = range(1, 10001)

    #sekwencyjna
    start = time.perf_counter()

    wyniki_zwykle = []
    for liczba in zakres:
        wyniki_zwykle.append(calculate_power_sum(liczba))

    koniec = time.perf_counter()
    czas_zwykle = koniec - start

    print(f"Czas sekwencyjnie: {czas_zwykle:.2f} s")


    #multiprocessing
    start = time.perf_counter()

    liczba_procesow = multiprocessing.cpu_count()

    with multiprocessing.Pool(processes=liczba_procesow) as pula:
        wyniki_procesy = pula.map(calculate_power_sum, zakres)

    koniec = time.perf_counter()
    czas_procesy = koniec - start

    print(f"Czas multiprocessing: {czas_procesy:.2f} s")

    print("Czy wyniki są takie same?", wyniki_zwykle == wyniki_procesy)

    if czas_procesy < czas_zwykle:
        print(f"Multiprocessing był szybszy o około {czas_zwykle - czas_procesy:.2f} s")
    else:
        print(f"Sekwencyjnie wyszło szybciej o około {czas_procesy - czas_zwykle:.2f} s")

Czas sekwencyjnie: 0.89 s
Czas multiprocessing: 1.25 s
Czy wyniki są takie same? True
Sekwencyjnie wyszło szybciej o około 0.36 s
